# Landing exploration — Cloud Provider Analytics (Group 11)

**Purpose.** Read every Landing source with PySpark, without modifying it, and record what is
actually there: grain, row counts, nulls, type inconsistencies and date coverage. This is the
evidence behind the source inventory and the quality rules in `docs/design.md`.

**What it produces** (written to `evidence/`):

| File | Content |
| :--- | :--- |
| `landing_profile.md` | Full report: every table below, rendered as Markdown. |
| `landing_source_profile.csv` | One row per source: format, rows, grain, duplicates, date range. |
| `landing_column_profile.csv` | One row per column: null count and null share. |

**How to run.** Put the Landing files under `data/landing/` (they are git-ignored and never
edited), then run all cells. Override the location with the `DATA_LANDING` environment variable
if your copy lives elsewhere.

**Reading strategy.** The CSV masters are read with every column as `string`, on purpose. Letting
Spark infer or cast types here would turn malformed values into silent nulls, and the whole point
of this notebook is to count them. Casting happens afterwards, explicitly, only where we profile
a numeric range. The events are read with the explicit superset schema from `docs/design.md`.

## 1. Setup

In [1]:
import os
from datetime import date
from pathlib import Path

from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType,
    IntegerType, LongType, TimestampType,
)

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
LANDING = Path(os.environ.get("DATA_LANDING", REPO_ROOT / "data" / "landing"))
EVIDENCE = REPO_ROOT / "evidence"
EVIDENCE.mkdir(exist_ok=True)

# On WSL the hostname resolves to a loopback address and Spark warns about it on every start.
# Binding explicitly keeps the local driver quiet and the saved outputs machine-neutral.
os.environ.setdefault("SPARK_LOCAL_IP", "127.0.0.1")

spark = (
    SparkSession.builder
    .appName("landing-exploration")
    .master("local[*]")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

def rel(path: Path) -> str:
    """Print paths relative to the repo so the saved outputs are not machine-specific."""
    try:
        return str(path.relative_to(REPO_ROOT))
    except ValueError:            # DATA_LANDING pointing outside the repo
        return str(path)

print("Spark   :", spark.version)
print("Landing :", rel(LANDING))
print("Evidence:", rel(EVIDENCE))
print("Run date:", date.today().isoformat())

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/10/03 20:20:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark   : 4.2.0
Landing : data/landing
Evidence: evidence
Run date: 2026-10-03


## 2. Read the sources

Seven CSV masters and one folder of JSONL event parts. `resources.csv` carries a quoted JSON
array in `tags_json`, so the reader is told how quotes are escaped; without that the embedded
commas would split the row.

In [2]:
CSV_SOURCES = {
    # source name          grain key(s)                 date column
    "customers_orgs":     (["org_id"],                  "signup_date"),
    "users":              (["user_id"],                 "created_at"),
    "resources":          (["resource_id"],             "created_at"),
    "support_tickets":    (["ticket_id"],               "created_at"),
    "marketing_touches":  (["touch_id"],                "timestamp"),
    "nps_surveys":        (["org_id", "survey_date"],   "survey_date"),
    "billing_monthly":    (["invoice_id"],              "month"),
}

def read_csv_as_text(name: str) -> DataFrame:
    """Every column as string, so no bad value is silently turned into a null on read."""
    return (
        spark.read
        .option("header", True)
        .option("quote", '"')
        .option("escape", '"')       # tags_json uses RFC-4180 doubled quotes
        .option("mode", "FAILFAST")  # a row we cannot even split must not pass unnoticed
        .csv(str(LANDING / f"{name}.csv"))
    )

csv_dfs = {name: read_csv_as_text(name).cache() for name in CSV_SOURCES}
for name, df in csv_dfs.items():
    print(f"{name:20s} rows={df.count():>6}  cols={len(df.columns):>2}")

customers_orgs       rows=    80  cols=11
users                rows=   800  cols= 7


resources            rows=   400  cols= 7
support_tickets      rows=  1000  cols= 8


marketing_touches    rows=  1500  cols= 7
nps_surveys          rows=    92  cols= 4


billing_monthly      rows=   240  cols= 8


In [3]:
# The superset schema from docs/design.md: one reader for v1 and v2 events.
# carbon_kg and genai_tokens are nullable, so v1 rows load with them as null.
# value stays a string because it arrives as a number, as text, or as null.
USAGE_EVENT_SCHEMA = StructType([
    StructField("event_id",           StringType(),    nullable=False),
    StructField("timestamp",          TimestampType(), nullable=False),
    StructField("org_id",             StringType(),    nullable=False),
    StructField("resource_id",        StringType(),    nullable=False),
    StructField("service",            StringType(),    nullable=False),
    StructField("region",             StringType(),    nullable=False),
    StructField("metric",             StringType(),    nullable=False),
    StructField("value",              StringType(),    nullable=True),
    StructField("unit",               StringType(),    nullable=True),
    StructField("cost_usd_increment", DoubleType(),    nullable=True),
    StructField("schema_version",     IntegerType(),   nullable=False),
    StructField("carbon_kg",          DoubleType(),    nullable=True),
    StructField("genai_tokens",       LongType(),      nullable=True),
    # Anything the schema cannot place lands here instead of being dropped.
    StructField("_corrupt_record",    StringType(),    nullable=True),
])

EVENT_PARTS = sorted(p.name for p in (LANDING / "usage_events_stream").glob("*.jsonl"))

events = (
    spark.read
    .schema(USAGE_EVENT_SCHEMA)
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .json(str(LANDING / "usage_events_stream" / "*.jsonl"))
    # source_file is the traceability column Bronze will keep.
    .withColumn("source_file", F.element_at(F.split(F.input_file_name(), "/"), -1))
    .cache()
)

n_events = events.count()
n_corrupt = events.filter(F.col("_corrupt_record").isNotNull()).count()
print(f"part files        : {len(EVENT_PARTS)}")
print(f"events            : {n_events:,}")
print(f"corrupt records   : {n_corrupt}")
print(f"null timestamps   : {events.filter(F.col('timestamp').isNull()).count()}")

part files        : 120
events            : 43,200
corrupt records   : 0
null timestamps   : 0


The superset schema reads all of it with zero corrupt records and zero unparseable timestamps,
which is the claim `docs/design.md` makes. Everything below profiles the content, not the shape.

## 3. Per-source profile

In [4]:
def dup_key_count(df: DataFrame, keys: list[str]) -> int:
    """How many key values appear more than once."""
    return df.groupBy(*keys).count().filter(F.col("count") > 1).count()

def null_col_count(df: DataFrame) -> int:
    row = df.agg(*[
        F.sum(F.col(c).isNull().cast("long")).alias(c) for c in df.columns
    ]).first()
    return sum(1 for c in df.columns if (row[c] or 0) > 0)

def date_range(df: DataFrame, col: str) -> str:
    lo, hi = df.agg(F.min(F.to_date(col)), F.max(F.to_date(col))).first()
    return "n/a" if lo is None else f"{lo} to {hi}"

source_profile = []
for name, (keys, date_col) in CSV_SOURCES.items():
    df = csv_dfs[name]
    source_profile.append({
        "source": f"{name}.csv",
        "format": "CSV",
        "files": 1,
        "rows": df.count(),
        "columns": len(df.columns),
        "grain_key": " + ".join(keys),
        "duplicate_keys": dup_key_count(df, keys),
        "cols_with_nulls": null_col_count(df),
        "date_column": date_col,
        "date_range": date_range(df, date_col),
    })

source_profile.append({
    "source": "usage_events_stream/*.jsonl",
    "format": "JSONL",
    "files": len(EVENT_PARTS),
    "rows": n_events,
    "columns": len(USAGE_EVENT_SCHEMA) - 1,   # _corrupt_record is not a source field
    "grain_key": "event_id",
    "duplicate_keys": dup_key_count(events, ["event_id"]),
    "cols_with_nulls": null_col_count(events.drop("_corrupt_record", "source_file")),
    "date_column": "timestamp",
    "date_range": date_range(events, "timestamp"),
})

SOURCE_COLS = ["source", "format", "files", "rows", "columns", "grain_key",
               "duplicate_keys", "cols_with_nulls", "date_column", "date_range"]
spark.createDataFrame(source_profile).select(*SOURCE_COLS).show(truncate=False)

+---------------------------+------+-----+-----+-------+--------------------+--------------+---------------+-----------+------------------------+
|source                     |format|files|rows |columns|grain_key           |duplicate_keys|cols_with_nulls|date_column|date_range              |
+---------------------------+------+-----+-----+-------+--------------------+--------------+---------------+-----------+------------------------+
|customers_orgs.csv         |CSV   |1    |80   |11     |org_id              |0             |1              |signup_date|2025-05-04 to 2025-07-02|
|users.csv                  |CSV   |1    |800  |7      |user_id             |0             |1              |created_at |2025-05-04 to 2025-08-11|
|resources.csv              |CSV   |1    |400  |7      |resource_id         |0             |1              |created_at |2025-05-04 to 2025-08-21|
|support_tickets.csv        |CSV   |1    |1000 |8      |ticket_id           |0             |2              |created_at |2025

In [5]:
# The grain claims below, measured rather than assumed.
print("orgs in customers_orgs      :", csv_dfs["customers_orgs"].select("org_id").distinct().count())
print("orgs appearing in events    :", events.select("org_id").distinct().count())
print("nps_surveys rows / orgs     :",
      csv_dfs["nps_surveys"].count(), "/", csv_dfs["nps_surveys"].select("org_id").distinct().count())
print("billing rows / orgs / months:",
      csv_dfs["billing_monthly"].count(),
      "/", csv_dfs["billing_monthly"].select("org_id").distinct().count(),
      "/", csv_dfs["billing_monthly"].select("month").distinct().count())

orgs in customers_orgs      : 80


orgs appearing in events    : 80


nps_surveys rows / orgs     : 92 / 60


billing rows / orgs / months: 240 / 80 / 3


Every master is unique on its own key and the events are unique on `event_id` in this dump.
That does **not** remove the need to dedupe: re-running an ingest would re-read the same part
files, so `event_id` is still the idempotency key.

Two grains are worth calling out. `nps_surveys` is *not* one row per org — 92 answers spread over
fewer orgs, several surveys each — so it cannot be joined to `customers_orgs` as if it were a
master. And `billing_monthly` is one invoice per org and month, three months only.

## 4. Column-level nulls

In [6]:
def column_nulls(name: str, df: DataFrame) -> list[dict]:
    total = df.count()
    row = df.agg(*[
        F.sum(F.col(c).isNull().cast("long")).alias(c) for c in df.columns
    ]).first()
    out = []
    for c in df.columns:
        nulls = int(row[c] or 0)
        out.append({
            "source": name,
            "column": c,
            "rows": total,
            "nulls": nulls,
            "null_pct": round(100.0 * nulls / total, 2) if total else 0.0,
        })
    return out

column_profile = []
for name, df in csv_dfs.items():
    column_profile += column_nulls(f"{name}.csv", df)
column_profile += column_nulls("usage_events_stream", events.drop("_corrupt_record", "source_file"))

COLUMN_COLS = ["source", "column", "rows", "nulls", "null_pct"]
nulls_df = spark.createDataFrame(column_profile).select(*COLUMN_COLS)

print("Columns that actually have nulls:")
nulls_df.filter(F.col("nulls") > 0).orderBy(F.col("null_pct").desc()).show(40, truncate=False)
print(f"columns profiled: {nulls_df.count()}  |  with nulls: {nulls_df.filter(F.col('nulls') > 0).count()}")

Columns that actually have nulls:


+-------------------+------------+-----+-----+--------+
|source             |column      |rows |nulls|null_pct|
+-------------------+------------+-----+-----+--------+
|usage_events_stream|genai_tokens|43200|40068|92.75   |
|billing_monthly.csv|credits     |240  |137  |57.08   |
|support_tickets.csv|csat        |1000 |254  |25.4    |
|usage_events_stream|carbon_kg   |43200|10800|25.0    |
|support_tickets.csv|resolved_at |1000 |240  |24.0    |
|resources.csv      |tags_json   |400  |83   |20.75   |
|nps_surveys.csv    |nps_score   |92   |19   |20.65   |
|users.csv          |last_login  |800  |139  |17.38   |
|customers_orgs.csv |nps_score   |80   |11   |13.75   |
|nps_surveys.csv    |comment     |92   |10   |10.87   |
|usage_events_stream|unit        |43200|2075 |4.8     |
|usage_events_stream|value       |43200|877  |2.03    |
+-------------------+------------+-----+-----+--------+



columns profiled: 65  |  with nulls: 12


These are the nulls the design has to survive. `unit`, `value`, `csat`, `resolved_at`, `credits`,
`last_login`, `tags_json` and both NPS columns are all expected to be missing on part of the data,
so no pipeline step may assume them present.

## 5. Usage events: schema versions and dirty types

In [7]:
print("Schema version split, and where the v2-only fields actually appear:")
events.groupBy("schema_version").agg(
    F.count("*").alias("events"),
    F.min(F.to_date("timestamp")).alias("first_day"),
    F.max(F.to_date("timestamp")).alias("last_day"),
    F.sum(F.col("carbon_kg").isNotNull().cast("long")).alias("carbon_kg_present"),
    F.sum(F.col("genai_tokens").isNotNull().cast("long")).alias("genai_tokens_present"),
).orderBy("schema_version").show()

print("genai_tokens is only ever populated for one service:")
events.filter(F.col("genai_tokens").isNotNull()).groupBy("service").count().show()

Schema version split, and where the v2-only fields actually appear:


+--------------+------+----------+----------+-----------------+--------------------+
|schema_version|events| first_day|  last_day|carbon_kg_present|genai_tokens_present|
+--------------+------+----------+----------+-----------------+--------------------+
|             1| 10800|2025-07-03|2025-07-17|                0|                   0|
|             2| 32400|2025-07-18|2025-08-31|            32400|                3132|
+--------------+------+----------+----------+-----------------+--------------------+

genai_tokens is only ever populated for one service:
+-------+-----+
|service|count|
+-------+-----+
|  genai| 3132|
+-------+-----+



In [8]:
# How `value` arrives. It is a string column, so we can tell the three cases apart:
# null, numeric text, and text that is not a number at all.
value_shape = events.select(
    F.when(F.col("value").isNull(), "null")
     .when(F.col("value").cast("double").isNotNull(), "numeric")
     .otherwise("non-numeric text").alias("value_kind")
).groupBy("value_kind").count().orderBy(F.col("count").desc())
value_shape.show()

print("value strings that would NOT survive a cast to double:",
      events.filter(F.col("value").isNotNull() & F.col("value").cast("double").isNull()).count())

+----------+-----+
|value_kind|count|
+----------+-----+
|   numeric|42323|
|      null|  877|
+----------+-----+

value strings that would NOT survive a cast to double: 0


In [9]:
# Reading `value` as a string erases how it was written in the JSON, so the raw lines are
# scanned separately to recover it. This is the type inconsistency the dataset warns about.
raw_lines = spark.read.text(str(LANDING / "usage_events_stream" / "*.jsonl")).withColumnRenamed("value", "line")

value_json_kind = raw_lines.select(
    F.when(F.col("line").rlike(r'"value":\s*null'), "null")
     .when(F.col("line").rlike(r'"value":\s*"'), "string (quoted in JSON)")
     .otherwise("number").alias("json_type")
).groupBy("json_type").count().orderBy(F.col("count").desc())
value_json_kind.show(truncate=False)

+-----------------------+-----+
|json_type              |count|
+-----------------------+-----+
|number                 |41014|
|string (quoted in JSON)|1309 |
|null                   |877  |
+-----------------------+-----+



There it is: 1,309 events write `value` as a quoted string and 41,014 write it as a JSON number,
for the same field and the same meaning. A single reader has to tolerate both, which is the
concrete case behind the Variety row of the 5V table.

The split matters for where the cast belongs. Every non-null `value` in this dump is castable, so
the string-in-Bronze rule costs us nothing today and protects the load the day a genuinely
non-numeric value shows up: it arrives as text, the Silver cast fails, and the row is quarantined
instead of silently becoming a null inside a sum.

In [10]:
print("cost_usd_increment distribution (the anomaly input):")
cost_stats = events.select(
    F.count("cost_usd_increment").alias("non_null"),
    F.round(F.min("cost_usd_increment"), 2).alias("min"),
    F.round(F.expr("percentile_approx(cost_usd_increment, 0.5)"), 2).alias("p50"),
    F.round(F.expr("percentile_approx(cost_usd_increment, 0.99)"), 2).alias("p99"),
    F.round(F.expr("percentile_approx(cost_usd_increment, 0.999)"), 2).alias("p999"),
    F.round(F.max("cost_usd_increment"), 2).alias("max"),
    F.sum((F.col("cost_usd_increment") < -0.01).cast("long")).alias("below_minus_0_01"),
)
cost_stats.show()

print("Conformance: service and region are consistent, no cleaning needed on these two.")
print("services:", sorted(r[0] for r in events.select("service").distinct().collect()))
print("regions :", sorted(r[0] for r in events.select("region").distinct().collect()))
print("metrics :", sorted(r[0] for r in events.select("metric").distinct().collect()))

cost_usd_increment distribution (the anomaly input):


+--------+-------+---+-----+------+------+----------------+
|non_null|    min|p50|  p99|  p999|   max|below_minus_0_01|
+--------+-------+---+-----+------+------+----------------+
|   43200|-154.46|1.0|16.69|128.74|317.43|             211|
+--------+-------+---+-----+------+------+----------------+

Conformance: service and region are consistent, no cleaning needed on these two.
services: ['analytics', 'compute', 'database', 'genai', 'networking', 'storage']


regions : ['ap-northeast', 'ap-south', 'eu-central', 'eu-west', 'sa-east', 'us-east', 'us-west']
metrics : ['cpu_hours', 'requests', 'storage_gb_hours']


The tail is the reason anomaly detection gets its own mart: p99 sits near 17 USD while the maximum
is above 300, so a fixed threshold would either miss real spikes or flag normal days. A relative
measure (z-score, MAD or percentile) is the right tool.

## 6. Late data — the finding that shapes the architecture

The part files are not time-ordered slices. Each one mixes events from almost the whole period, so
if we replay them in filename order as micro-batches, most events arrive long after the watermark
would have closed their window.

The simulation below walks the files in order, tracks the maximum event time seen so far (exactly
what a watermark does), and counts the events whose own timestamp is older than that maximum minus
the watermark delay.

In [11]:
per_file = events.groupBy("source_file").agg(
    F.count("*").alias("events"),
    F.min("timestamp").alias("min_ts"),
    F.max("timestamp").alias("max_ts"),
).orderBy("source_file")

span = per_file.select(
    F.round(F.avg(F.datediff(F.to_date("max_ts"), F.to_date("min_ts"))), 1).alias("avg_span_days"),
    F.min(F.datediff(F.to_date("max_ts"), F.to_date("min_ts"))).alias("min_span_days"),
    F.max(F.datediff(F.to_date("max_ts"), F.to_date("min_ts"))).alias("max_span_days"),
    F.round(F.avg("events"), 1).alias("avg_events_per_file"),
)
print("Each part file spans almost the entire dataset period:")
span.show()
per_file.show(5, truncate=False)

Each part file spans almost the entire dataset period:


+-------------+-------------+-------------+-------------------+
|avg_span_days|min_span_days|max_span_days|avg_events_per_file|
+-------------+-------------+-------------+-------------------+
|         59.0|           59|           59|              360.0|
+-------------+-------------+-------------+-------------------+

+----------------------+------+-------------------+-------------------+
|source_file           |events|min_ts             |max_ts             |
+----------------------+------+-------------------+-------------------+
|events_part_0000.jsonl|360   |2025-07-03 00:08:00|2025-08-31 16:32:00|
|events_part_0001.jsonl|360   |2025-07-03 02:38:00|2025-08-31 23:51:00|
|events_part_0002.jsonl|360   |2025-07-03 14:16:00|2025-08-31 20:47:00|
|events_part_0003.jsonl|360   |2025-07-03 00:48:00|2025-08-31 23:51:00|
|events_part_0004.jsonl|360   |2025-07-03 10:43:00|2025-08-31 20:40:00|
+----------------------+------+-------------------+-------------------+
only showing top 5 rows


In [12]:
# Replay the parts in filename order and count what a watermark would drop.
ordered = (
    events.select("source_file", "timestamp")
    .collect()
)
by_file: dict[str, list] = {}
for r in ordered:
    by_file.setdefault(r["source_file"], []).append(r["timestamp"])

DELAYS_DAYS = [1, 7, 30]
late = {d: 0 for d in DELAYS_DAYS}
total = 0
max_seen = None
for part in EVENT_PARTS:                      # micro-batch = one part file, in order
    batch = by_file.get(part, [])
    for ts in batch:
        total += 1
        if max_seen is not None:
            behind = (max_seen - ts).total_seconds() / 86400.0
            for d in DELAYS_DAYS:
                if behind > d:
                    late[d] += 1
    if batch:                                 # watermark only advances between batches
        batch_max = max(batch)
        max_seen = batch_max if max_seen is None else max(max_seen, batch_max)

watermark_rows = [{
    "watermark": f"{d} day{'s' if d > 1 else ''}",
    "late_events": late[d],
    "late_pct": round(100.0 * late[d] / total, 1),
} for d in DELAYS_DAYS]

WATERMARK_COLS = ["watermark", "late_events", "late_pct"]
print(f"events replayed: {total:,} across {len(EVENT_PARTS)} micro-batches")
spark.createDataFrame(watermark_rows).select(*WATERMARK_COLS).show(truncate=False)

events replayed: 43,200 across 120 micro-batches


+---------+-----------+--------+
|watermark|late_events|late_pct|
+---------+-----------+--------+
|1 day    |42118      |97.5    |
|7 days   |37831      |87.6    |
|30 days  |21421      |49.6    |
+---------+-----------+--------+



**Consequence for the design.** A stateful streaming aggregation would throw away most of the
data: even a 30-day watermark loses about half the events, and a realistic one-day watermark loses
nearly all of them. So streaming is used for *append-only ingestion* into Bronze/Silver, and the
daily Gold marts are recomputed in **batch** over Silver, where a late event simply lands in the
day it belongs to. This is the concrete reason the architecture is Lambda-style rather than pure
Kappa — the data forced it, not the syllabus.

## 7. Quality findings that become rules

Each row below is a check the pipeline will run, with the count it returns on today's Landing.
This is the baseline: the same checks on a later load should not get worse without a reason.

### 7.1 Events

The rule set is **impute-first, quarantine-last**. A row goes to Quarantine only when the data
contradicts itself or cannot be placed at all; a row that is merely incomplete but unambiguous is
repaired and marked. Each rule reports the share of events *and* the share of total cost it
touches, because a rule that rejects 5% of events can silently remove 5% of revenue.

In [13]:
orgs = csv_dfs["customers_orgs"]
resources = csv_dfs["resources"]
tickets = csv_dfs["support_tickets"]
billing = csv_dfs["billing_monthly"]
nps = csv_dfs["nps_surveys"]

org_ids = [r["org_id"] for r in orgs.select("org_id").collect()]
res_ids = [r["resource_id"] for r in resources.select("resource_id").collect()]

# metric determines unit. Checked here, before any imputation relies on it.
print("unit values observed per metric:")
events.groupBy("metric", "unit").count().orderBy("metric", F.col("count").desc()).show(truncate=False)

print("distinct non-null units per metric (must be 1 for imputation to be safe):")
events.filter(F.col("unit").isNotNull()).groupBy("metric").agg(
    F.countDistinct("unit").alias("distinct_units"),
    F.min("unit").alias("unit"),
    F.count("*").alias("rows_with_unit"),
).orderBy("metric").show()

unit values observed per metric:


+----------------+--------+-----+
|metric          |unit    |count|
+----------------+--------+-----+
|cpu_hours       |hours   |10187|
|cpu_hours       |NULL    |505  |
|requests        |count   |18596|
|requests        |NULL    |916  |
|storage_gb_hours|gb_hours|12342|
|storage_gb_hours|NULL    |654  |
+----------------+--------+-----+

distinct non-null units per metric (must be 1 for imputation to be safe):


+----------------+--------------+--------+--------------+
|          metric|distinct_units|    unit|rows_with_unit|
+----------------+--------------+--------+--------------+
|       cpu_hours|             1|   hours|         10187|
|        requests|             1|   count|         18596|
|storage_gb_hours|             1|gb_hours|         12342|
+----------------+--------------+--------+--------------+



Each metric uses exactly one unit and never a different one, so `unit` carries no information that
`metric` does not already determine. That is what makes imputation safe rather than a guess — and
it is also why a *contradicting* unit is a different matter: there, two fields disagree and the row
cannot be trusted.

In [14]:
UNIT_BY_METRIC = {"requests": "count", "cpu_hours": "hours", "storage_gb_hours": "gb_hours"}

expected_unit = F.create_map(
    *[lit for k, v in UNIT_BY_METRIC.items() for lit in (F.lit(k), F.lit(v))]
)
checked = events.withColumn("expected_unit", expected_unit[F.col("metric")])

total_cost = checked.agg(F.sum("cost_usd_increment")).first()[0]
print(f"total cost_usd_increment in Landing: {total_cost:,.2f} USD")

QUARANTINE, KEEP = "quarantine", "keep"
RULES = [
    ("event_id is null",
     F.col("event_id").isNull(), QUARANTINE, "cannot dedupe or trace"),
    ("metric not in the known set",
     ~F.col("metric").isin(list(UNIT_BY_METRIC)), QUARANTINE, "cannot map to a feature"),
    ("unit contradicts metric",
     F.col("unit").isNotNull() & (F.col("unit") != F.col("expected_unit")),
     QUARANTINE, "two fields disagree"),
    ("value present but not castable to double",
     F.col("value").isNotNull() & F.col("value").cast("double").isNull(),
     QUARANTINE, "unusable measurement"),
    ("org_id not in customers_orgs",
     ~F.col("org_id").isin(org_ids), QUARANTINE, "unjoinable"),
    ("resource_id not in resources",
     ~F.col("resource_id").isin(res_ids), QUARANTINE, "unjoinable"),
    ("unit is null, metric is known",
     F.col("unit").isNull() & F.col("metric").isin(list(UNIT_BY_METRIC)),
     KEEP, "impute unit from metric, set unit_imputed = true"),
    ("value is null",
     F.col("value").isNull(), KEEP, "null (not zero) in usage sums; cost still counted"),
    ("cost_usd_increment < -0.01",
     F.col("cost_usd_increment") < -0.01, KEEP, "cost_anomaly_flag = true"),
]

agg = checked.agg(*[
    expr for i, (_, pred, _, _) in enumerate(RULES) for expr in (
        F.sum(pred.cast("long")).alias(f"n_{i}"),
        F.sum(F.when(pred, F.col("cost_usd_increment")).otherwise(0.0)).alias(f"c_{i}"),
    )
]).first()

event_rules = []
for i, (label, _, zone, action) in enumerate(RULES):
    n = int(agg[f"n_{i}"] or 0)
    c = float(agg[f"c_{i}"] or 0.0)
    event_rules.append({
        "rule": label,
        "zone": zone,
        "events": n,
        "pct_events": round(100.0 * n / n_events, 2),
        "pct_cost": round(100.0 * c / total_cost, 2),
        "action": action,
    })

EVENT_RULE_COLS = ["rule", "zone", "events", "pct_events", "pct_cost", "action"]
spark.createDataFrame(event_rules).select(*EVENT_RULE_COLS).show(60, truncate=False)

total cost_usd_increment in Landing: 147,433.98 USD


+----------------------------------------+----------+------+----------+--------+-------------------------------------------------+
|rule                                    |zone      |events|pct_events|pct_cost|action                                           |
+----------------------------------------+----------+------+----------+--------+-------------------------------------------------+
|event_id is null                        |quarantine|0     |0.0       |0.0     |cannot dedupe or trace                           |
|metric not in the known set             |quarantine|0     |0.0       |0.0     |cannot map to a feature                          |
|unit contradicts metric                 |quarantine|0     |0.0       |0.0     |two fields disagree                              |
|value present but not castable to double|quarantine|0     |0.0       |0.0     |unusable measurement                             |
|org_id not in customers_orgs            |quarantine|0     |0.0       |0.0     |unj

In [15]:
# The quarantine baseline is the share of events matching ANY quarantine rule.
quarantine_pred = None
for _, pred, zone, _ in RULES:
    if zone == QUARANTINE:
        quarantine_pred = pred if quarantine_pred is None else (quarantine_pred | pred)

q = checked.agg(
    F.sum(quarantine_pred.cast("long")).alias("events"),
    F.sum(F.when(quarantine_pred, F.col("cost_usd_increment")).otherwise(0.0)).alias("cost"),
).first()
quarantine_events = int(q["events"] or 0)
quarantine_cost = float(q["cost"] or 0.0)
quarantine_pct = round(100.0 * quarantine_events / n_events, 2)
quarantine_cost_pct = round(100.0 * quarantine_cost / total_cost, 2)

imputed    = next(r for r in event_rules if r["rule"].startswith("unit is null"))
null_value = next(r for r in event_rules if r["rule"] == "value is null")
dup_events = dup_key_count(events, ["event_id"])

print(f"QUARANTINE baseline : {quarantine_events} events "
      f"({quarantine_pct}% of events, {quarantine_cost_pct}% of cost)")
print(f"unit imputed        : {imputed['events']} events ({imputed['pct_events']}% of events, "
      f"{imputed['pct_cost']}% of cost) -- kept, flagged unit_imputed")
print(f"null value kept     : {null_value['events']} events ({null_value['pct_events']}% of events, "
      f"{null_value['pct_cost']}% of cost) -- cost counted, usage contributes null")
print(f"duplicate event_id  : {dup_events} (dedupe still required for re-runs)")

QUARANTINE baseline : 0 events (0.0% of events, 0.0% of cost)
unit imputed        : 2075 events (4.8% of events, 4.92% of cost) -- kept, flagged unit_imputed
null value kept     : 877 events (2.03% of events, 2.01% of cost) -- cost counted, usage contributes null
duplicate event_id  : 0 (dedupe still required for re-runs)


**Nothing in this dump is quarantined.** Every rejecting rule returns zero: no null or duplicate
`event_id`, no unknown metric, no unit contradicting its metric, no uncastable value, and every
`org_id` and `resource_id` resolves against the masters.

The two populations a stricter design would have thrown away are kept instead:

- **2,075 events (4.80% of events, 4.92% of cost)** have a null `unit`. Quarantining them would
  understate every FinOps mart by roughly 5%. They are repaired from `metric` and flagged
  `unit_imputed = true`, so the repair stays auditable downstream. Of these, 2,038 (4.82% of cost)
  also carry a `value`; the remaining 37 have neither `unit` nor `value`, and still get the unit
  imputed — a missing measurement is not a reason to lose the cost attached to it.
- **877 events (2.03% of events, 2.01% of cost)** have a null `value`. The cost on those rows is
  real and is counted; only the usage measurement is unknown, so it contributes `null` rather than
  `0` to usage sums. Treating it as zero would bias every average downward.

This is why objective **O6** is set against a measured quarantine baseline of **0%**, with headroom
for genuinely malformed data, instead of the 4.72% a quarantine-first reading produced.

### 7.2 Masters and billing

In [16]:
master_checks = [
    ("customers_orgs: nps_score null",
     orgs.filter(F.col("nps_score").isNull()).count()),
    ("customers_orgs: nps_score outside 0..100",
     orgs.filter((F.col("nps_score").cast("double") < 0) |
                 (F.col("nps_score").cast("double") > 100)).count()),
    ("nps_surveys: nps_score null",
     nps.filter(F.col("nps_score").isNull()).count()),
    ("support_tickets: resolved_at null (still open)",
     tickets.filter(F.col("resolved_at").isNull()).count()),
    ("support_tickets: csat null",
     tickets.filter(F.col("csat").isNull()).count()),
    ("support_tickets: severity = critical",
     tickets.filter(F.lower("severity") == "critical").count()),
    ("support_tickets: sla_breached true",
     tickets.filter(F.lower("sla_breached").isin("true", "1", "yes")).count()),
    ("billing_monthly: credits null",
     billing.filter(F.col("credits").isNull()).count()),
    ("billing_monthly: subtotal negative",
     billing.filter(F.col("subtotal").cast("double") < 0).count()),
]
MASTER_CHECK_COLS = ["check", "rows"]
spark.createDataFrame(
    [{"check": c, "rows": n} for c, n in master_checks]
).select(*MASTER_CHECK_COLS).show(truncate=False)

print("customers_orgs.nps_score observed range (cast to double):")
orgs.select(F.min(F.col("nps_score").cast("double")).alias("min"),
            F.max(F.col("nps_score").cast("double")).alias("max")).show()

print("billing_monthly is multi-currency, so revenue has to be converted before it is summed:")
billing.groupBy("currency").agg(
    F.count("*").alias("invoices"),
    F.round(F.min(F.col("exchange_rate_to_usd").cast("double")), 4).alias("min_rate"),
    F.round(F.max(F.col("exchange_rate_to_usd").cast("double")), 4).alias("max_rate"),
).orderBy(F.col("invoices").desc()).show()

print("Billing months covered:")
billing.groupBy("month").count().orderBy("month").show()

+----------------------------------------------+----+
|check                                         |rows|
+----------------------------------------------+----+
|customers_orgs: nps_score null                |11  |
|customers_orgs: nps_score outside 0..100      |17  |
|nps_surveys: nps_score null                   |19  |
|support_tickets: resolved_at null (still open)|240 |
|support_tickets: csat null                    |254 |
|support_tickets: severity = critical          |56  |
|support_tickets: sla_breached true            |95  |
|billing_monthly: credits null                 |137 |
|billing_monthly: subtotal negative            |13  |
+----------------------------------------------+----+

customers_orgs.nps_score observed range (cast to double):
+-----+-----+
|  min|  max|
+-----+-----+
|-38.0|101.0|
+-----+-----+

billing_monthly is multi-currency, so revenue has to be converted before it is summed:


+--------+--------+--------+--------+
|currency|invoices|min_rate|max_rate|
+--------+--------+--------+--------+
|     USD|     160|  0.8546|  1.1179|
|     ARS|      51|  0.0013|  0.0016|
|     EUR|      29|  0.9981|  1.1981|
+--------+--------+--------+--------+

Billing months covered:
+----------+-----+
|     month|count|
+----------+-----+
|2025-06-01|   80|
|2025-07-01|   80|
|2025-08-01|   80|
+----------+-----+



`nps_score` is out of range on both sides (negative and above 100), so it is noise, not a scale we
can trust — it gets a validity rule rather than an average. Billing being in three currencies with
a per-invoice rate is why `revenue_by_org_month` must convert to USD at the invoice's own rate
instead of applying one rate per month.

## 8. Write the evidence files

In [17]:
def markdown_table(rows: list[dict], cols: list[str]) -> str:
    head = "| " + " | ".join(cols) + " |"
    sep  = "| " + " | ".join(["---"] * len(cols)) + " |"
    body = ["| " + " | ".join(str(r.get(c, "")) for c in cols) + " |" for r in rows]
    return "\n".join([head, sep, *body])

def write_csv(path: Path, rows: list[dict], cols: list[str]) -> None:
    import csv
    with path.open("w", newline="") as fh:
        # lineterminator pinned to LF so re-runs do not show up as whole-file diffs
        w = csv.DictWriter(fh, fieldnames=cols, lineterminator="\n")
        w.writeheader()
        w.writerows(rows)
    print(f"wrote {path.relative_to(REPO_ROOT)}  ({len(rows)} rows)")

write_csv(EVIDENCE / "landing_source_profile.csv", source_profile, SOURCE_COLS)
write_csv(EVIDENCE / "landing_column_profile.csv",
          sorted(column_profile, key=lambda r: (r["source"], r["column"])), COLUMN_COLS)

wrote evidence/landing_source_profile.csv  (8 rows)
wrote evidence/landing_column_profile.csv  (65 rows)


In [18]:
value_rows      = [r.asDict() for r in value_shape.collect()]
value_json_rows = [r.asDict() for r in value_json_kind.collect()]
version_rows = [r.asDict() for r in events.groupBy("schema_version").agg(
    F.count("*").alias("events"),
    F.sum(F.col("carbon_kg").isNotNull().cast("long")).alias("carbon_kg_present"),
    F.sum(F.col("genai_tokens").isNotNull().cast("long")).alias("genai_tokens_present"),
).orderBy("schema_version").collect()]
cost_row = cost_stats.first().asDict()
span_row = span.first().asDict()

report = f"""# Landing exploration — evidence

Generated by `notebooks/01_landing_exploration.ipynb` on {date.today().isoformat()}
with PySpark {spark.version}. Source: `data/landing/` (read-only, unmodified).

Landing holds **{len(CSV_SOURCES)} CSV masters** and **{len(EVENT_PARTS)} JSONL event parts**
({n_events:,} events), read with **0 corrupt records** using the superset schema in
`docs/design.md`.

## 1. Per-source profile

{markdown_table(source_profile, SOURCE_COLS)}

## 2. Columns with nulls

{markdown_table([r for r in sorted(column_profile, key=lambda r: -r["null_pct"]) if r["nulls"] > 0], COLUMN_COLS)}

Full per-column listing, including columns with no nulls: `landing_column_profile.csv`.

## 3. Event schema versions

{markdown_table([{k: v for k, v in r.items()} for r in version_rows],
                ["schema_version", "events", "carbon_kg_present", "genai_tokens_present"])}

`genai_tokens` appears only on `service = genai`. Both v2-only fields are null for every v1 row,
which is why the superset schema declares them nullable.

## 4. How `value` arrives

As written in the JSON (recovered from the raw lines), the same field uses two different types:

{markdown_table(value_json_rows, ["json_type", "count"])}

As read through the superset schema, by castability:

{markdown_table(value_rows, ["value_kind", "count"])}

Every non-null value is castable to double today. It is still read as `string` in Bronze so that a
future non-numeric value fails the Silver cast and is quarantined, instead of becoming a null
inside a sum.

## 5. cost_usd_increment distribution

{markdown_table([cost_row], list(cost_row.keys()))}

## 6. Late data

Each part file spans almost the whole period, so the files are not time-ordered slices:

{markdown_table([span_row], list(span_row.keys()))}

Replaying the {len(EVENT_PARTS)} parts in filename order as micro-batches, the share of events a
watermark would treat as late:

{markdown_table(watermark_rows, WATERMARK_COLS)}

**Consequence.** Stateful streaming windows would discard most of the data. Streaming does
append-only ingestion into Bronze/Silver; the daily Gold marts are recomputed in batch over
Silver. This is the measured reason for a Lambda-style design.

## 7. Quality checks and their baseline counts

### Events

Impute-first, quarantine-last. `pct_cost` is the share of total `cost_usd_increment` each rule
touches — the figure that decides whether a rule may reject rows at all.

{markdown_table(event_rules, EVENT_RULE_COLS)}

`metric` determines `unit` 1:1 in the data, which is what makes imputation safe instead of a guess.

**Quarantine baseline: {quarantine_events} events ({quarantine_pct}% of events, {quarantine_cost_pct}% of cost).**
Nothing here is rejected. Kept and flagged instead: {imputed["events"]} events
({imputed["pct_events"]}% of events, {imputed["pct_cost"]}% of cost) with an imputed `unit`, and
{null_value["events"]} events ({null_value["pct_events"]}%, {null_value["pct_cost"]}% of cost) with a
null `value` whose cost is still counted. Duplicate `event_id`: {dup_events} — dedupe is still
required because a re-run would re-read the same part files.

### Masters and billing

{markdown_table([{"check": c, "rows": n} for c, n in master_checks], MASTER_CHECK_COLS)}
"""

out = EVIDENCE / "landing_profile.md"
out.write_text(report)
print(f"wrote {out.relative_to(REPO_ROOT)}  ({len(report.splitlines())} lines)")

wrote evidence/landing_profile.md  (140 lines)


## 9. What this changes in the design

1. **The superset schema holds.** 43,200 events across both versions read with zero corrupt
   records, and the v2-only fields are null exactly where expected.
2. **Streaming ingests, batch aggregates.** The late-data measurement is the deciding evidence:
   windowed streaming aggregation is not viable on this data.
3. **Partition events by `event_date` only.** Roughly 60 days of data; adding service or org as a
   partition would shatter it into tiny files for no gain at this volume.
4. **Quality rules are quarantine-first.** The rules that fire (negative cost, `unit` missing next
   to a present `value`) affect a small share of rows. Dropping files over them would lose far
   more than it protects.
5. **Convert currency per invoice.** Billing is USD, ARS and EUR with a per-invoice rate, so
   revenue is summed only after conversion.
6. **Volume is the weakest of the 5Vs here.** 13 MB total. The justification rests on velocity,
   variety and veracity, with volume projected to real provider scale.

In [19]:
spark.stop()
print("Spark session stopped.")

Spark session stopped.
